In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')


plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Loading dataset...")
try:

    df = pd.read_csv('water_potability.csv')
    print("Dataset loaded successfully!")
except FileNotFoundError:
    print("Dataset file not found. Please ensure your CSV file is in the correct path.")
    print("Trying common file names...")
    try:

        df = pd.read_csv('water_quality.csv')
    except FileNotFoundError:
        try:
            df = pd.read_csv('water_data.csv')
        except FileNotFoundError:
            try:
                df = pd.read_csv('dataset.csv')
            except FileNotFoundError:
                print("Please make sure your dataset file is in the same directory.")
                print("Common file names: 'water_potability.csv', 'water_quality.csv', 'water_data.csv'")
                exit()

print(f"Dataset shape: {df.shape}")


print("\nDataset columns:")
print(df.columns.tolist())

print("\nFirst 5 rows of the dataset:")
print(df.head())


print("\n" + "="*50)
print("1. EXPLORATORY DATA ANALYSIS")
print("="*50)


print("\nDataset Info:")
print(df.info())
print(f"\nMissing values:\n{df.isnull().sum()}")

print("\nHandling missing values...")
df.fillna(df.mean(), inplace=True)


target_column = None
possible_targets = ['Potability', 'potability', 'Target', 'target', 'Class', 'class', 'Quality', 'quality']

for col in possible_targets:
    if col in df.columns:
        target_column = col
        break

if target_column is None:

    target_column = df.columns[-1]
    print(f"Target column not found using common names. Using last column: {target_column}")

print(f"Using '{target_column}' as target variable")

print("\nDataset Overview:")
print(df.describe())


print(f"\nTarget Distribution:\n{df[target_column].value_counts()}")
print(f"Potability Ratio: {df[target_column].value_counts(normalize=True)}")


plt.figure(figsize=(15, 12))

plt.subplot(3, 3, 1)
df[target_column].value_counts().plot(kind='bar', color=['skyblue', 'lightcoral'])
plt.title('Target Variable Distribution (Potability)')
plt.xlabel('Potability (0: Non-Potable, 1: Potable)')
plt.ylabel('Count')
plt.xticks(rotation=0)


feature_columns = [col for col in df.columns if col != target_column]
num_features = min(4, len(feature_columns))

for i in range(num_features):
    plt.subplot(3, 3, i+2)
    df[feature_columns[i]].hist(bins=30, alpha=0.7, color=plt.cm.Set3(i))
    plt.title(f'{feature_columns[i]} Distribution')
    plt.xlabel(feature_columns[i])
    plt.ylabel('Frequency')


for i in range(min(2, len(feature_columns))):
    plt.subplot(3, 3, i+6)
    sns.boxplot(x=target_column, y=feature_columns[i], data=df)
    plt.title(f'{feature_columns[i]} vs Potability')

plt.tight_layout()
plt.show()


print("\n" + "="*50)
print("2. CORRELATION ANALYSIS")
print("="*50)


plt.figure(figsize=(12, 8))
correlation_matrix = df.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

print("\n" + "="*50)
print("3. DATA PREPROCESSING")
print("="*50)

X = df.drop(target_column, axis=1)
y = df[target_column]

print(f"Original dataset shape: {X.shape}")
print(f"Original class distribution: {np.bincount(y)}")


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed!")

print("\nApplying SMOTE for class balance...")
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"Dataset shape after SMOTE: {X_train_balanced.shape}")
print(f"Class distribution after SMOTE: {np.bincount(y_train_balanced)}")


print("\n" + "="*50)
print("4. MODEL TRAINING AND EVALUATION")
print("="*50)


models = {
    'Logistic Regression': LogisticRegression(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}


results = {}


for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train_balanced, y_train_balanced)


    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]


    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'roc_auc': roc_auc,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }

    print(f"{name} Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")


print("\n" + "="*50)
print("5. RESULTS COMPARISON AND VISUALIZATION")
print("="*50)

results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[model]['accuracy'] for model in results],
    'Precision': [results[model]['precision'] for model in results],
    'Recall': [results[model]['recall'] for model in results],
    'F1-Score': [results[model]['f1'] for model in results],
    'ROC-AUC': [results[model]['roc_auc'] for model in results]
})

print("\nModel Performance Comparison:")
print(results_df.round(4))


plt.figure(figsize=(15, 10))


plt.subplot(2, 3, 1)
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
x_pos = np.arange(len(metrics))

for i, model in enumerate(results.keys()):
    model_metrics = [
        results[model]['accuracy'],
        results[model]['precision'],
        results[model]['recall'],
        results[model]['f1'],
        results[model]['roc_auc']
    ]
    plt.plot(metrics, model_metrics, marker='o', label=model, linewidth=2)

plt.title('Model Performance Comparison')
plt.xlabel('Metrics')
plt.ylabel('Score')
plt.legend()
plt.grid(True, alpha=0.3)


plt.subplot(2, 3, 2)
for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result['probabilities'])
    plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC = {result["roc_auc"]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1)
plt.title('ROC Curves')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend()
plt.grid(True, alpha=0.3)


for i, (name, result) in enumerate(results.items(), 3):
    plt.subplot(2, 3, i)
    cm = confusion_matrix(y_test, result['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Potable', 'Potable'],
                yticklabels=['Non-Potable', 'Potable'])
    plt.title(f'Confusion Matrix - {name}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

plt.tight_layout()
plt.show()


print("\n" + "="*50)
print("6. FEATURE IMPORTANCE ANALYSIS")
print("="*50)


plt.figure(figsize=(15, 5))


plt.subplot(1, 3, 1)
rf_importance = results['Random Forest']['model'].feature_importances_
feature_names = X.columns
rf_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_importance
}).sort_values('importance', ascending=True)

plt.barh(rf_importance_df['feature'], rf_importance_df['importance'], color='skyblue')
plt.title('Random Forest - Feature Importance')
plt.xlabel('Importance Score')

plt.subplot(1, 3, 2)
xgb_importance = results['XGBoost']['model'].feature_importances_
xgb_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': xgb_importance
}).sort_values('importance', ascending=True)

plt.barh(xgb_importance_df['feature'], xgb_importance_df['importance'], color='lightgreen')
plt.title('XGBoost - Feature Importance')
plt.xlabel('Importance Score')


plt.subplot(1, 3, 3)
combined_importance = (rf_importance + xgb_importance) / 2
combined_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': combined_importance
}).sort_values('importance', ascending=True)

plt.barh(combined_importance_df['feature'], combined_importance_df['importance'], color='orange')
plt.title('Combined - Feature Importance')
plt.xlabel('Importance Score')

plt.tight_layout()
plt.show()


print("\nTop 5 Most Important Features (Combined):")
top_features = combined_importance_df.tail(5)
for _, row in top_features.iterrows():
    print(f"{row['feature']}: {row['importance']:.4f}")

print("\n" + "="*50)
print("7. DETAILED MODEL ANALYSIS")
print("="*50)


best_model_name = max(results, key=lambda x: results[x]['f1'])
best_model = results[best_model_name]['model']

print(f"\nBest Performing Model: {best_model_name}")
print(f"Best F1-Score: {results[best_model_name]['f1']:.4f}")

print(f"\nDetailed Classification Report for {best_model_name}:")
best_predictions = results[best_model_name]['predictions']
print(classification_report(y_test, best_predictions,
                          target_names=['Non-Potable', 'Potable']))


print("\n" + "="*50)
print("8. PROJECT SUMMARY AND CONCLUSIONS")
print("="*50)

print("""
PROJECT SUMMARY:

1. DATA PREPROCESSING:
   - Handled missing values using mean imputation
   - Applied StandardScaler for feature normalization
   - Used SMOTE to address class imbalance

2. MODELS USED:
   - Logistic Regression (Baseline with explainability)
   - Random Forest (Ensemble method with high accuracy)
   - XGBoost (Gradient boosting with best performance)

3. KEY FINDINGS:
   - Multi-parameter analysis provides reliable water quality classification
   - Feature importance reveals most critical water quality parameters
   - Machine learning enables accurate potability prediction

4. PRACTICAL APPLICATIONS:
   - Rapid water quality assessment for authorities
   - Early warning system for water contamination
   - Data-driven decision making for water management
""")


results_df.to_csv('water_quality_model_results.csv', index=False)
print("\nResults saved to 'water_quality_model_results.csv'")


feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'RF_Importance': rf_importance,
    'XGB_Importance': xgb_importance,
    'Combined_Importance': combined_importance
}).sort_values('Combined_Importance', ascending=False)

feature_importance_df.to_csv('feature_importance_results.csv', index=False)
print("Feature importance results saved to 'feature_importance_results.csv'")

print("\n" + "="*50)
print("PROJECT COMPLETED SUCCESSFULLY!")
print("="*50)

print("\nFINAL MODEL PERFORMANCE SUMMARY:")
print("="*40)
print(results_df.round(4))


from google.colab import files
import pandas as pd
import io

print("Please upload your water_potability.csv file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))

print(" SUCCESS! Dataset loaded!")
print(f"Shape: {df.shape}")
print(f"First 5 rows:")
print(df.head())


Loading dataset...
Dataset file not found. Please ensure your CSV file is in the correct path.
Trying common file names...
Please make sure your dataset file is in the same directory.
Common file names: 'water_potability.csv', 'water_quality.csv', 'water_data.csv'


NameError: name 'df' is not defined

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
